## Example 1 — the import, the installed version and the cuFile library it loaded

In [ ]:
import importlib.metadata
import cufile
from cufile import bindings

installed = importlib.metadata.version("cufile-python")
# Upstream forgot to bump `cufile.__version__` (it still says 0.1.0 in the 0.2.0 release), so the
# distribution's own metadata is the version of record.
print("cufile-python", installed, "(module __version__:", cufile.__version__ + ")")
assert sorted(cufile.__all__) == ["CuFile", "CuFileDriver"], cufile.__all__
assert bindings.libcufile._name == "libcufile.so", bindings.libcufile._name
for symbol in ("cuFileDriverOpen", "cuFileHandleRegister", "cuFileRead", "cuFileWrite", "cuFileBufRegister"):
    assert getattr(bindings.libcufile, symbol) is not None, symbol
print("OK -- cufile imported and", bindings.libcufile._name, "exports the cuFile calls it binds")

## Example 2 — open the cuFile driver and read a file into host memory

In [ ]:
import ctypes, os, tempfile
from cufile import CuFile, CuFileDriver

driver = CuFileDriver()  # cuFileDriverOpen(); raises RuntimeError on a cuFile error
assert driver is CuFileDriver(), "CuFileDriver must be one object per process"
payload = bytes(range(256)) * 64
path = os.path.join(tempfile.mkdtemp(prefix="cufile_nb_"), "payload.bin")
with open(path, "wb") as handle:
    handle.write(payload)
buffer = ctypes.create_string_buffer(len(payload))
with CuFile(path, "r") as file:
    assert file.is_open
    count = file.read(ctypes.cast(buffer, ctypes.c_void_p), len(payload))
assert not file.is_open
assert count == len(payload) and buffer.raw == payload, (count, buffer.raw[:8])
print(f"OK -- the driver opened and cuFileRead returned {count} bytes that match the file")

## Example 3 — write from host memory, then read back at an offset

In [ ]:
import ctypes, os, tempfile
from cufile import CuFile

payload = os.urandom(64 * 1024)
source = ctypes.create_string_buffer(payload, len(payload))
path = os.path.join(tempfile.mkdtemp(prefix="cufile_nb_"), "written.bin")
with CuFile(path, "w+") as file:
    written = file.write(ctypes.cast(source, ctypes.c_void_p), len(payload))
assert written == len(payload), written
with open(path, "rb") as handle:
    assert handle.read() == payload, "the file does not hold what cuFileWrite was given"
window = ctypes.create_string_buffer(4096)
with CuFile(path, "r") as file:
    count = file.read(ctypes.cast(window, ctypes.c_void_p), 4096, file_offset=8192)
assert count == 4096 and window.raw == payload[8192:8192 + 4096]
print(f"OK -- cuFileWrite wrote {written} bytes; a 4096-byte read at offset 8192 matches")

## Example 4 — GPU memory to a file and back, in the same process as torch

In [ ]:
import ctypes, os, tempfile, torch
from cufile import CuFile

assert torch.cuda.is_available(), "requires-gpu cell ran but torch sees no CUDA device"
source = torch.arange(1 << 20, dtype=torch.uint8, device="cuda")
target = torch.zeros_like(source)
path = os.path.join(tempfile.mkdtemp(prefix="cufile_nb_"), "tensor.bin")
with CuFile(path, "w+") as file:
    written = file.write(ctypes.c_void_p(source.data_ptr()), source.numel())
with CuFile(path, "r") as file:
    count = file.read(ctypes.c_void_p(target.data_ptr()), target.numel())
assert written == count == source.numel() == os.path.getsize(path), (written, count, os.path.getsize(path))
assert torch.equal(source, target), "the tensor read back from the file differs"
print(f"OK -- {count} bytes went GPU -> file -> GPU and match (libcufile and torch share this process)")

## Example 5 — failures arrive as Python exceptions, not crashes

In [ ]:
import ctypes, os, tempfile
from cufile import CuFile

folder = tempfile.mkdtemp(prefix="cufile_nb_")
buffer = ctypes.create_string_buffer(16)
missing = CuFile(os.path.join(folder, "does_not_exist.bin"), "r")
try:
    missing.open()
    raise AssertionError("opening a missing file for reading was accepted")
except FileNotFoundError as error:
    refused_open = type(error).__name__
try:
    missing.read(ctypes.cast(buffer, ctypes.c_void_p), 16)
    raise AssertionError("a read on a file that is not open was accepted")
except OSError as error:
    assert "not open" in str(error), error
with open(os.path.join(folder, "small.bin"), "wb") as handle:
    handle.write(b"data")
try:
    with CuFile(os.path.join(folder, "small.bin"), "r") as file:
        raise ValueError("raised inside the with block")
except ValueError:
    assert not file.is_open, "the file stayed open after an error inside the with block"
print(f"OK -- a missing file raises {refused_open}, a closed file raises OSError, an error in `with` closes it")